In [1]:
import pandas as pd

# 1. Cargar los datos crudos desde el entorno de Colab
df_clientes = pd.read_csv('clientes.csv')
df_productos = pd.read_csv('productos.csv')
df_ventas = pd.read_csv('ventas.csv')
df_gastos = pd.read_csv('gastos.csv')
df_presupuesto = pd.read_csv('presupuesto.csv')

# 2. Limpieza y Transformación
# Convertir fechas a formato datetime de Pandas para evitar errores en Power BI
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])

# Limpiar posibles strings vacíos o espacios accidentales en IDs
df_clientes['cliente_id'] = df_clientes['cliente_id'].astype(str).str.strip()
df_productos['producto_id'] = df_productos['producto_id'].astype(str).str.strip()
df_ventas['cliente_id'] = df_ventas['cliente_id'].astype(str).str.strip()
df_ventas['producto_id'] = df_ventas['producto_id'].astype(str).str.strip()

# 3. Validación de Calidad de Datos
print("--- Resumen de Calidad de Datos ---")
print(f"Duplicados en Ventas: {df_ventas.duplicated().sum()}")
print(f"Duplicados en Clientes: {df_clientes.duplicated().sum()}")
print(f"Nulos en Ventas:\n{df_ventas.isnull().sum()[df_ventas.isnull().sum() > 0]}")

--- Resumen de Calidad de Datos ---
Duplicados en Ventas: 8
Duplicados en Clientes: 0
Nulos en Ventas:
canal    15
dtype: int64


In [2]:
!pip install gspread-dataframe -q

from google.colab import auth
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

# 1. Autenticación (Te pedirá permisos para acceder a tu Google Drive)
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Crear el Data Warehouse en tu Drive
nombre_dw = 'DW_Andina_Comercial'
try:
    sh = gc.open(nombre_dw)
    print("Archivo encontrado. Actualizando datos...")
except gspread.exceptions.SpreadsheetNotFound:
    sh = gc.create(nombre_dw)
    print(f"Nuevo archivo '{nombre_dw}' creado en la raíz de tu Google Drive.")

# 3. Función para exportar DataFrames a pestañas individuales
def exportar_a_sheets(df, sheet_name):
    try:
        worksheet = sh.worksheet(sheet_name)
        worksheet.clear() # Limpiar datos viejos si los hay
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=sheet_name, rows=1, cols=1)

    # Manejar nulos para que Google Sheets no arroje error
    df_export = df.fillna("")
    set_with_dataframe(worksheet, df_export)
    print(f" Datos exportados con éxito a la pestaña: {sheet_name}")

# 4. Ejecutar la exportación
exportar_a_sheets(df_clientes, 'Clientes')
exportar_a_sheets(df_productos, 'Productos')
exportar_a_sheets(df_ventas, 'Ventas')
exportar_a_sheets(df_gastos, 'Gastos')
exportar_a_sheets(df_presupuesto, 'Presupuesto')

# Borrar la hoja por defecto si está vacía
try:
    hoja_default = sh.worksheet('Sheet1')
    sh.del_worksheet(hoja_default)
except:
    pass

Nuevo archivo 'DW_Andina_Comercial' creado en la raíz de tu Google Drive.
 Datos exportados con éxito a la pestaña: Clientes
 Datos exportados con éxito a la pestaña: Productos
 Datos exportados con éxito a la pestaña: Ventas
 Datos exportados con éxito a la pestaña: Gastos
 Datos exportados con éxito a la pestaña: Presupuesto
